# Penguin GPT fine-tuning

This notebook downloads Wikipedia pages about penguins and fine-tunes a GPT model
on the collected text. It assumes you have Python 3.12 and a working PyTorch install.


## Imports


In [9]:
from pathlib import Path

import requests
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)


## Configuration


In [10]:
BASE_MODEL = "gpt2-medium"  # Try gpt2-medium if you have enough GPU memory
OUTPUT_DIR = Path("artifacts/penguin-gpt")
DATA_DIR = Path("data")
RAW_TEXT_PATH = DATA_DIR / "penguins.txt"
MAX_LENGTH = 128
EPOCHS = 5
BATCH_SIZE = 2
LEARNING_RATE = 5e-5
SEED = 42

PAGES = [
    "Penguin",
    "Emperor_penguin",
    "King_penguin",
    "Adelie_penguin",
    "Gentoo_penguin",
    "Chinstrap_penguin",
    "Little_penguin",
    "Rockhopper_penguin",
    "Macaroni_penguin",
    "African_penguin",
    "Magellanic_penguin",
    "Humboldt_penguin",
    "Galapagos_penguin",
]


## Download Wikipedia text


In [11]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch_page(title: str) -> str:
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
        "format": "json",
    }
    resp = requests.get(url, params=params, timeout=30, headers={"User-Agent": "otus-gpt/1.0"})
    resp.raise_for_status()
    data = resp.json()
    pages = data.get("query", {}).get("pages", {})
    if not pages:
        raise ValueError(f"No pages found for {title}")
    page = next(iter(pages.values()))
    text = page.get("extract", "")
    if not text:
        raise ValueError(f"Empty extract for {title}")
    return text

if RAW_TEXT_PATH.exists():
    raw_text = RAW_TEXT_PATH.read_text(encoding="utf-8")
else:
    texts = []
    for title in PAGES:
        print(f"Downloading {title}...")
        try:
            texts.append(fetch_page(title))
        except Exception as exc:
            print(f"Skip {title}: {exc}")
    raw_text = "\n\n".join(texts)
    RAW_TEXT_PATH.write_text(raw_text, encoding="utf-8")

len(raw_text)


250116

## Clean and split into paragraphs


In [12]:
def normalize(text: str) -> list[str]:
    paragraphs = []
    buffer = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            if buffer:
                paragraphs.append(" \n".join(buffer).replace(" \n", " "))
                buffer = []
            continue
        if line.startswith("==") or line.startswith("References"):
            continue
        buffer.append(line)
    if buffer:
        paragraphs.append(" \n".join(buffer).replace(" \n", " "))
    paragraphs = [p for p in paragraphs if len(p) > 60]
    return paragraphs

paragraphs = normalize(raw_text)
len(paragraphs)


231

## Tokenize


In [13]:
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

encoded = tokenizer(
    paragraphs,
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
    return_tensors="pt",
)

class TokenizedDataset(torch.utils.data.Dataset):
    def __init__(self, batch):
        self.batch = batch

    def __len__(self) -> int:
        return self.batch["input_ids"].shape[0]

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.batch["input_ids"][idx],
            "attention_mask": self.batch["attention_mask"][idx],
        }

dataset = TokenizedDataset(encoded)
len(dataset)


c:\Users\katan\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\katan\.cache\huggingface\hub\models--gpt2-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


231

## Train


In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=20,
    save_strategy="no",
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
  3%|▎         | 20/580 [00:53<24:22,  2.61s/it]

{'loss': 3.1719, 'grad_norm': 15.816493034362793, 'learning_rate': 4.827586206896552e-05, 'epoch': 0.17}


  7%|▋         | 40/580 [01:45<23:23,  2.60s/it]

{'loss': 3.202, 'grad_norm': 20.951011657714844, 'learning_rate': 4.655172413793104e-05, 'epoch': 0.34}


 10%|█         | 60/580 [02:37<22:38,  2.61s/it]

...

 97%|█████████▋| 560/580 [23:04<00:50,  2.53s/it]

{'loss': 1.4175, 'grad_norm': 20.081079483032227, 'learning_rate': 1.724137931034483e-06, 'epoch': 4.83}


100%|██████████| 580/580 [23:55<00:00,  2.48s/it]


{'loss': 1.4155, 'grad_norm': 17.387001037597656, 'learning_rate': 0.0, 'epoch': 5.0}
{'train_runtime': 1435.7278, 'train_samples_per_second': 0.804, 'train_steps_per_second': 0.404, 'train_loss': 2.0536157312064334, 'epoch': 5.0}


('artifacts\\penguin-gpt\\tokenizer_config.json',
 'artifacts\\penguin-gpt\\special_tokens_map.json',
 'artifacts\\penguin-gpt\\vocab.json',
 'artifacts\\penguin-gpt\\merges.txt',
 'artifacts\\penguin-gpt\\added_tokens.json',
 'artifacts\\penguin-gpt\\tokenizer.json')

## Generate a sample


In [ ]:
prompt = "who is a African penguin?"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


who is a African penguin? African penguin species are split into two major subspecies: African and African white. The African penguin is a separate species from the subspecies Eudyptes chrysocome, the African rockhopper penguin and the southern African penguin. The African rockhopper penguin is a monotypic species in which the northern and southern subspecies share a highly divergent phylogen
